In [48]:
import pandas as pd
import numpy as np
import altair as alt

In [49]:
fossil_data = pd.read_csv('fossil-fuel-consumption-per-person-by-fuel.csv')
print(fossil_data.shape)

fossil_data.head()

(11456, 6)


,Entity,Code,Year,Gas,Oil,Coal
0,Afghanistan,AFG,1980,46.600067,325.44598,55.173412
1,Afghanistan,AFG,1981,77.112430,409.19400,63.935123
2,Afghanistan,AFG,1982,139.584560,433.30948,80.549310
3,Afghanistan,AFG,1983,140.522900,615.18536,81.090790
4,Afghanistan,AFG,1984,137.104250,601.86346,80.754930


In [50]:
fossil_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11456 entries, 0 to 11455
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Entity  11456 non-null  object 
 1   Code    11456 non-null  object 
 2   Year    11456 non-null  int64  
 3   Gas     11103 non-null  float64
 4   Oil     11391 non-null  float64
 5   Coal    11328 non-null  float64
dtypes: float64(3), int64(1), object(2)
memory usage: 537.1+ KB


In [51]:
#Checking missing values and zeros in each column
pd.DataFrame({'missing' : fossil_data[['Gas','Oil','Coal']].isna().sum(),
             'zeros' : (fossil_data[['Gas', 'Oil','Coal']]== 0).sum()})

,missing,zeros
Gas,353,4897
Oil,65,126
Coal,128,4352


In [52]:
#How mny entities have data in each year?
fossil_data.groupby('Year')['Entity'].nunique().loc[[1800,1900,1964,1965,1979,1980,1990,2000,2024,2025]]

#Entities with code is not ISO code (regions, income groups, and former country)
fossil_data[fossil_data['Code'].str.len()!=3].groupby(['Entity','Code'])['Year'].agg(['min','max'])

,,min,max
Entity,Code,,
Africa,OWID_AFR,1965,2025
Asia,OWID_ASI,1965,2025
Czechoslovakia,OWID_CZS,1980,1992
East Germany,OWID_GDR,1980,1990
Europe,OWID_EUR,1965,2025
European Union (27),OWID_EU27,1965,2025
High-income countries,OWID_HIC,1965,2025
Kosovo,OWID_KOS,2008,2024
Low-income countries,OWID_LIC,1980,2024


In [53]:
#Largest values in each type of fuel column
for fuel in ['Gas','Oil','Coal']:
  print(fuel)
  print(fossil_data.nlargest(3,fuel)[['Entity','Year',fuel]].to_string(index=False), '\n')

Gas
Entity  Year       Gas
 Qatar  1993 288358.53
 Qatar  1994 279747.06
 Qatar  1992 278111.47 

Oil
   Entity  Year       Oil
Gibraltar  2009 1846352.0
Gibraltar  2007 1841370.6
Gibraltar  2005 1793412.6 

Coal
      Entity  Year      Coal
  Luxembourg  1965 96194.445
  Luxembourg  1970 92327.266
East Germany  1985 90641.020 



###Source:

The data was obtained from Our World in Data's chart, Fossil fuel consumption per capita by fuel. The data is compiled from the Energy Institute's Statistical Review of World Energy (2026) and the U.S. Energy Information Administration's International Energy Data (2026), with long-run historical estimates from Etead & Luciani (1991), Smil (2017), and the UK National Infrastructure Commission (2020).

Link: https://ourworldindata.org/grapher/fossil-fuel-consumption-per-person-by-fuel

###What does the data contain:

The data contains 11,456 rows and 6 columns. Each row is one entity in one year. The entity is the name of a country, region, or group. Code is the three-letter ISO country code. Year runs from 1800 to 2025. The other three columns are Gas, Oil, and Coal; they provide an average amount of each fuel consumed per person that year. In the file, the unit was not specified. From Our World in Data, it seems the unit is kilowatt-hours (kWh), which seems to match the magnitude of the data itself.

For this project, we chose to focus on the United States, which contains 61 rows, from 1965 to 2025. There are no missing or NA values.

###Challenge and detail to be awared of:

1. Mixed entity types: Countries, continents, income groups, and the World are stacked in the same columns. Summing or averaging across rows could result in double counting.

2. Per person only: The data only shows consumption per person, not the total consumption, so small countries could rank above larger ones.

3. Missing values: Gas has 353 missing values, coal has 128 missing values. Gas also has 4897 zeros, and coal has 4352 zeros. Zeros could mean that a country did not use fuel or that the consumption is too small to report, so zeros should be treated carefully.

#Data cleaning

In [54]:
#Clean column names (units : kWh per person)
fossil_data = fossil_data.rename(columns= {'Entity': 'entity', 'Code':'code','Year':'year',
                                           'Gas':'gas_kwh','Oil':'oil_kwh','Coal':'coal_kwh'})

#Filter out only data coming from United States ,China, and European Union:
countries_list = ['United States','China','United Kingdom','Brazil','South Africa','Thailand','Germany','Norway','Japan','India']
filter_data = fossil_data[fossil_data['entity'].isin(countries_list) & (fossil_data['year']>=2010)].copy().reset_index(drop=True)
print(filter_data.shape)

filter_data.groupby(['entity','code'])['year'].agg(['min','max', 'count'])


(160, 6)


,,min,max,count
entity,code,,,
Brazil,BRA,2010,2025,16
China,CHN,2010,2025,16
Germany,DEU,2010,2025,16
India,IND,2010,2025,16
Japan,JPN,2010,2025,16
Norway,NOR,2010,2025,16
South Africa,ZAF,2010,2025,16
Thailand,THA,2010,2025,16
United Kingdom,GBR,2010,2025,16


In [55]:
#Check US rows for missing value or NA
fuels = ['gas_kwh', 'oil_kwh', 'coal_kwh']
print('missing values:', filter_data[fuels].isna().sum().to_dict())
print('zeros:', (filter_data[fuels] == 0).sum().to_dict())
print('duplicate entity year rows:', filter_data.duplicated(subset = ['entity','year']).sum())

missing values: {'gas_kwh': 0, 'oil_kwh': 0, 'coal_kwh': 0}
zeros: {'gas_kwh': 0, 'oil_kwh': 0, 'coal_kwh': 0}
duplicate entity year rows: 0


In [56]:
#Total fossil fuel use and each fuel's share of the total:
filter_data['total_kwh'] = filter_data[fuels].sum(axis=1)
for fuel in fuels:
  filter_data[fuel.replace('_kwh', '_share')] = filter_data[fuel] / filter_data['total_kwh']
filter_data[filter_data['year'].isin([2010,2025])].set_index(['entity','year']).drop(columns= 'code').round(3)

gas_kwh    oil_kwh   coal_kwh  total_kwh  gas_share  \
entity         year                                                          
Brazil         2010   1423.671   6762.346    868.332   9054.349      0.157   
               2025   1559.987   6771.116    772.286   9103.389      0.171   
China          2010    805.543   3858.123  15049.661  19713.327      0.041   
               2025   3120.789   6638.954  18093.281  27853.023      0.112   
Germany        2010  10895.737  16796.102  11086.998  38778.837      0.281   
               2025   9339.156  13873.402   4938.336  28150.894      0.332   
India          2010    474.166   1486.484   2692.880   4653.530      0.102   
               2025    454.749   2066.838   4377.882   6899.469      0.066   
Japan          2010   7796.272  19905.213  10552.965  38254.450      0.204   
               2025   7343.389  14687.906   9834.394  31865.689      0.230   
Norway         2010   8251.430  25497.344   1812.399  35561.173      0.232   
               2025   6048.741  18522.130   1524.933  26095.804      0.232   
South Africa   2010    787.493   5777.862  20032.975  26598.330      0.030   
               2025    722.951   4207.449  15639.960  20570.360      0.035   
Thailand       2010   6300.075   7791.620   2617.180  16708.875      0.377   
               2025   6604.669   8995.031   2304.712  17904.412      0.369   
United Kingdom 2010  15632.176  14051.708   5631.306  35315.190      0.443   
               2025   8793.729  10828.539    198.242  19820.510      0.444   
United States  2010  20838.027  31809.334  18647.496  71294.857      0.292   
               2025  26303.111  29134.710   6982.992  62420.813      0.421   

                     oil_share  coal_share  
entity         year                         
Brazil         2010      0.747       0.096  
               2025      0.744       0.085  
China          2010      0.196       0.763  
               2025      0.238       0.650  
Germany        2010      0.433       0.286  
               2025      0.493       0.175  
India          2010      0.319       0.579  
               2025      0.300       0.635  
Japan          2010      0.520       0.276  
               2025      0.461       0.309  
Norway         2010      0.717       0.051  
               2025      0.710       0.058  
South Africa   2010      0.217       0.753  
               2025      0.205       0.760  
Thailand       2010      0.466       0.157  
               2025      0.502       0.129  
United Kingdom 2010      0.398       0.159  
               2025      0.546       0.010  
United States  2010      0.446       0.262  
               2025      0.467       0.112

In [57]:
#Percent change from 2010 to 2023 per country:
percent_change = filter_data.pivot( index='year', columns = 'entity', values = fuels + ['total_kwh'])
((percent_change.loc[2023]/ percent_change.loc[2010]-1)*100).round(1).unstack()

#Checking highest and lowest year for each fuel along with the total share:
filter_data.groupby('entity').apply(lambda g: pd.Series({
    'peak year': g.loc[g['total_kwh'].idxmax(), 'year'], 'peak (kWh)': round(g['total_kwh'].max()),
    'lowest year': g.loc[g['total_kwh'].idxmin(), 'year'], 'lowest (kWh)': round(g['total_kwh'].min())}))

/tmp/ipykernel_8025/2343480697.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  filter_data.groupby('entity').apply(lambda g: pd.Series({


,peak year,peak (kWh),lowest year,lowest (kWh)
entity,,,,
Brazil,2014,10751,2020,8258
China,2025,27853,2010,19713
Germany,2013,38837,2024,28077
India,2024,6976,2010,4654
Japan,2012,41425,2025,31866
Norway,2010,35561,2024,25910
South Africa,2010,26598,2022,19814
Thailand,2018,19393,2010,16709
United Kingdom,2010,35315,2025,19821


In [58]:
#Tidy data format and ready for Altair:
data_alt = filter_data.melt(id_vars = ['entity','year'], value_vars = fuels, var_name = 'fuel',
                               value_name = 'kwh_per_person')
data_alt['fuel']= data_alt['fuel'].map({'gas_kwh':'Gas','oil_kwh':'Oil','coal_kwh':'Coal'})
print(data_alt.shape)
data_alt.head()

(480, 4)


,entity,year,fuel,kwh_per_person
0,Brazil,2010,Gas,1423.6709
1,Brazil,2011,Gas,1407.3514
2,Brazil,2012,Gas,1654.6718
3,Brazil,2013,Gas,1933.8524
4,Brazil,2014,Gas,2031.9089


###Cleaning data:

I renamed the columns to make the units appear clearer and filtered the data to 10 countries to match the other data. This leaves around 144 rows ( 16 years for each of the 10 countries). Each country has its own ISO code; this helps prevent duplicates across the data. I also added the total fossil fuel column and each fuel's share of the total, which shows countries of different sizes and income on a comparable scale; the data was reshaped and tidied up for easy incorporation with Altair later. For EDA, this dataset has one categorical feature, one time feature (year), and three numeric features. The code column is not needed if just using this dataset alone, but might be useful when merging with other datasets.

#Patterns worth investigating and visualization goals

From the data above, the fuel mix shows how differently these countries are changing. Coal is collapsing in Western countries, falling about 93% per person in the UK, 66% in the US, and 53% in Germany, while it is still maintained in Asia: 64% in India, 20% in China. Even though Japan is in Asia, it is among the wealthiest countries; the consumption of coal here is unchanged, while oil consumption is down 25%. Gas did replace coal in the US, but fell sharply in the UK and Germany, so their drops in coal were not just offset by gas. Brazil and Egypt have mixes unlike other countries. I would want to investigate which countries are replacing coal with gas, or cutting fossil fuels overall, and whether rising use in India and China is slowing down or not.

To help demonstrate the patterns, the data would need to be merged with other data. The dataset could be merged with ISO codes and year. Population would turn per person into total consumption instead. The energy mix would show what replaces fossil fuels; $CO_2$ emissions per person would show the climate impact of each fuel mix. Key events should also be accounted for, for instance, the pandemic in 2020 and the 2022 energy crisis in Europe and Japan.

For visualizations, I would make a dot or slope chart for total fossil fuel use per person ranging from 2010 to 2023 for the ten countries. This data could be portrayed in multiple line charts of gas, oil, and coal per person, with one panel for each country. A scatter plot for fossil fuel use against GDP could also be considered if there are data to support it.